In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "data")
ARTIFACT_PATH = os.path.join(BASE_DIR, "artifacts")

os.makedirs(ARTIFACT_PATH, exist_ok=True)

In [2]:
movies = pd.read_csv(os.path.join(DATA_PATH, "movie.csv"))
ratings = pd.read_csv(os.path.join(DATA_PATH, "rating.csv"),
                       usecols=["userId", "movieId", "rating"])

In [3]:
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

top_movies = ratings["movieId"].value_counts().head(300).index
ratings = ratings[ratings["movieId"].isin(top_movies)]

top_users = ratings["userId"].value_counts().head(800).index
ratings = ratings[ratings["userId"].isin(top_users)]

movies = movies[movies["movieId"].isin(ratings["movieId"])].reset_index(drop=True)

print(len(movies), len(ratings))

300 211550


In [4]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

cosine_sim = cosine_similarity(tfidf_matrix)

In [5]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

In [6]:
user_item_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0).astype(np.float32)

item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

In [7]:
valid_users = ratings["userId"].unique()

print("Valid users count:", len(valid_users))

Valid users count: 800


In [8]:
print("Saving artifacts...")

joblib.dump(movies, os.path.join(ARTIFACT_PATH, "movies.pkl"))
joblib.dump(tfidf, os.path.join(ARTIFACT_PATH, "tfidf.pkl"))
joblib.dump(cosine_sim.astype(np.float16), os.path.join(ARTIFACT_PATH, "content_model.pkl"))
joblib.dump(indices, os.path.join(ARTIFACT_PATH, "indices.pkl"))
joblib.dump(item_similarity_df.astype(np.float16), os.path.join(ARTIFACT_PATH, "collaborative_model.pkl"))
joblib.dump(valid_users, os.path.join(ARTIFACT_PATH, "valid_users.pkl"))

print("ALL FILES SAVED SUCCESSFULLY")

Saving artifacts...
ALL FILES SAVED SUCCESSFULLY


In [9]:
valid_users

array([    91,    294,    427,    586,    741,    775,    903,    982,
         1376,   1411,   1507,   1748,   1849,   2158,   2261,   2397,
         2669,   3171,   3318,   3487,   3576,   3629,   3743,   3907,
         3990,   4129,   4222,   4358,   4507,   4529,   4594,   4696,
         5576,   5623,   5768,   5843,   5900,   6099,   6232,   6373,
         6636,   6873,   6921,   6976,   6978,   7051,   7201,   7321,
         7370,   7699,   7828,   7858,   8152,   8405,   8441,   8568,
         8932,   8966,   9087,   9116,   9145,   9446,   9544,   9545,
         9562,  10271,  10303,  10443,  10560,  10678,  10721,  10916,
        10989,  11074,  11246,  11348,  11560,  11865,  11900,  12131,
        12239,  12644,  12767,  12793,  12802,  13064,  13155,  13753,
        13849,  14280,  14519,  14551,  14705,  15194,  15203,  15266,
        15486,  15529,  15601,  15617,  15720,  16398,  16676,  16840,
        16865,  16938,  17163,  17250,  17918,  18138,  18191,  18280,
      

In [11]:
import pickle
pickle.dump(movies, open("artifacts/movies1.pkl", "wb"))

In [12]:
joblib.dump(ratings, "artifacts/ratings.pkl")

['artifacts/ratings.pkl']